In [1]:
suppressMessages({
library(SoupX)
library(Seurat)
library(DropletUtils)
})

set.seed(123)
args = commandArgs(T)


Warning message:
“package ‘GenomeInfoDb’ was built under R version 4.2.3”


In [ ]:
input = '01.af_web.Raw_Filter/'
outdir = '01.self.matrix/02.af_SoupX.Matrix/'

for ( lib_i in df$sample ){
        sample = gsub('-.*$','',lib_i)
        sample_library = lib_i
        path = df[which(df$sample == lib_i ),'path']
        toc <- Read10X( paste0(path,'/FilterMatrix')  ,gene.column=1) #FilterMatrix
        print(length(unique(rownames(toc))))
    
        tod <-  Read10X(  paste0(path,'/RawMatrix')  ,gene.column=1) #RawMatrix
        print(length(unique(rownames(tod))))
        tod <- tod[rownames(toc),]     
        
        all <- toc
        all <- CreateSeuratObject(all)
        all <- NormalizeData(all, normalization.method = "LogNormalize", scale.factor = 10000)
        all <- FindVariableFeatures(all, selection.method = "vst", nfeatures = 3000)
        all.genes <- rownames(all)
        all <- ScaleData(all, features = all.genes)

        all <- RunPCA(all, features = VariableFeatures(all), npcs = 40, verbose = F)
        all <- FindNeighbors(all, dims = 1:30)
        all <- FindClusters(all, resolution = 0.5)
        all <- RunUMAP(all, dims = 1:30)

        matx <- all@meta.data
        sc = SoupChannel(tod, toc)
        sc = setClusters(sc, setNames(matx$seurat_clusters, rownames(matx)))
        
        tryCatch(
        {sc = autoEstCont(sc)},
        error=function(e) {
                
                sc <<- setContaminationFraction(sc, 0.2)       
                print("autoEstCont Error !")})
        out = adjustCounts(sc)
        
        outpath = paste0(outdir,"/",sample_library,"/")
        dir.create(outpath,recursive = TRUE)
        DropletUtils:::write10xCounts(outpath, out ,version="3",overwrite = T)

        print(lib_i)
}

[1] 45816
[1] 50667


Centering and scaling data matrix

Computing nearest neighbor graph

Computing SNN



Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 32271
Number of edges: 854622

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.8855
Number of communities: 14
Elapsed time: 5 seconds


Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”
11:01:09 UMAP embedding parameters a = 0.9922 b = 1.112

11:01:09 Read 32271 rows and found 30 numeric columns

11:01:09 Using Annoy for neighbor search, n_neighbors = 30

11:01:09 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----|

*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
|

11:01:13 Writing NN index file to temp file /tmp/RtmpdNSd25/file1d213b7098c

11:01:13 Searching Annoy index using 1 thread, search_k = 3000

11:01:25 Annoy recall = 100%

11:01:27 Commencing smooth kNN distance calibration using 1 thread
 with target n_neighbors =

[1] "###########################################################################################################"
[1] "FCD22-1"
[1] 46254
[1] 51184


Centering and scaling data matrix

